# 03 specialist agents

**Owner:** Thomas. Original notebook cells: 53, 54.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# THOMAS: Specialist workflow integration

from collections import Counter

ROLE_WORKFLOWS = {
    "Owner": {
        "Scheduling",
        "Insurance Verification",
        "Optical Orders"
    },
    "Front Desk": {
        "Scheduling",
        "Insurance Verification",
        "Optical Orders"
    },
    "Optician": {
        "Optical Orders"
    }
}

WORKFLOW_OPERATIONS = {
    "Scheduling": {
        "GetAppointments",
        "GetNoShowRisk"
    },
    "Insurance Verification": {
        "GetVerificationStatus"
    },
    "Optical Orders": {
        "GetOfficeFrameOrders",
        "GetOfficeContactLensOrders"
    }
}

WORKFLOW_SCOPES = {
    "Scheduling": {"appointments.read"},
    "Insurance Verification": {"insurance.read"},
    "Optical Orders": {"orders.read"}
}

# Temporary Thomas-side fixtures.
# Replace with Daniel's final records when available.
TEMP_INSURANCE_RECORDS = [
    {
        "patient_id": 101,
        "verification_status": "Verified",
        "exception": None
    },
    {
        "patient_id": 102,
        "verification_status": "Needs Review",
        "exception": "Verification incomplete"
    }
]


class CapstoneWorkflowGateway(CrystalGateway):

    def __init__(self, staff_role, workflow, subject):
        if workflow not in ROLE_WORKFLOWS.get(
            staff_role, set()
        ):
            raise AgentError("Workflow access denied.")

        workflow_principal = Principal(
            subject,
            "demo-office",
            frozenset(WORKFLOW_SCOPES[workflow])
        )

        super().__init__(
            SPEC,
            workflow_principal,
            demo=True
        )

        self.workflow = workflow
        self.staff_role = staff_role
        self.read_trace = []

        # Limit each specialist to its approved operations.
        self.operations = {
            name: op
            for name, op in self.operations.items()
            if name in WORKFLOW_OPERATIONS[workflow]
        }

        if workflow == "Scheduling":
            self.operations["GetNoShowRisk"] = {
                "summary": "Read demo appointment no-show risk.",
                "x-umbrella-required-scope": "appointments.read",
                "parameters": []
            }

        if workflow == "Insurance Verification":
            self.operations["GetVerificationStatus"] = {
                "summary": "Read synthetic verification records.",
                "x-umbrella-required-scope": "insurance.read",
                "parameters": []
            }

    def _page(self, operation, params):
        op = self.authorize(operation)
        self.validate(op, params)

        if operation == "GetNoShowRisk":
            result = {
                "success": True,
                "data": [{
                    "scheduleId": 2,
                    "status": "Scheduled",
                    "data_mode": "demo",
                    **scheduling_risk(demo_booking_features)
                }]
            }

        elif operation == "GetVerificationStatus":
            result = {
                "success": True,
                "data": [
                    dict(record)
                    for record in TEMP_INSURANCE_RECORDS
                ]
            }

        else:
            result = super()._page(operation, params)

        self.read_trace.append(operation)
        return result


def create_workflow_agent(staff_role, workflow, subject):
    restricted_gateway = CapstoneWorkflowGateway(
        staff_role,
        workflow,
        subject
    )

    workflow_agent = EyeStyleAgent(
        client,
        MODEL,
        restricted_gateway
    )

    return workflow_agent, restricted_gateway


print("Specialist workflow gateways configured.")

In [ ]:
# THOMAS: Integrated Scheduling Agent test

scheduling_agent, scheduling_gateway = (
    create_workflow_agent(
        staff_role="Front Desk",
        workflow="Scheduling",
        subject="demo-front-desk"
    )
)

start = time.perf_counter() if "time" in globals() else None

import time
start = time.perf_counter()

scheduling_result = scheduling_agent.ask(
    "Use GetNoShowRisk to review synthetic appointment 2. "
    "Provide the actual model risk score, priority, "
    "and recommended staff action. "
    "Do not contact the patient or modify any records."
)

elapsed = time.perf_counter() - start

print("=== SCHEDULING AGENT RESULT ===")
print(scheduling_result["answer"])

print("\nExecution time:", round(elapsed, 3), "seconds")

print("\nData sources accessed:")
print(scheduling_gateway.read_trace)

print("\nAudit:")
print(json.dumps(scheduling_result["audit"], indent=2))

assert "GetNoShowRisk" in scheduling_gateway.read_trace